# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # The metadata object
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Access all record sets and their @id fields
print("Available Record Sets:")
record_sets = dataset.record_sets  # List of RecordSet objects
for record_set in record_sets:
    print(f"@id: {record_set.id} | name: {record_set.name}")

# For each record set, print its fields and @id's
print("\nRecord Set Fields:")
for record_set in record_sets:
    print(f"\nRecordSet: {record_set.name} (@id: {record_set.id})")
    print("Fields:")
    for field in record_set.fields:
        print(f"  @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# For this dataset, use the main data table record set (choose the first as example)
main_record_set_id = record_set_ids[0]
print(f"Loading records from RecordSet: {main_record_set_id}")

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {rs_id}")
    else:
        print(f"No records found for {rs_id}")

# Inspect DataFrame columns for the main record set
df = dataframes[main_record_set_id]
print("Columns in main DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select numeric field for analysis
# Let's auto-detect potential numeric fields (float/int), or set a default field if detected
numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")
else:
    # Manually fallback to a typical field name for mass spectrometry
    # Replace with the detected column name if available
    numeric_field = 'Coverage (%)' if 'Coverage (%)' in df.columns else df.columns[0]
    print(f"Fallback to numeric field: {numeric_field}")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a typical category (e.g., 'Description' or similar)
group_field_candidates = [col for col in df.columns if 'desc' in col.lower() or 'type' in col.lower()]
group_field = group_field_candidates[0] if group_field_candidates else df.columns[0]

if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the selected numeric field
plt.figure(figsize=(8, 5))
df[numeric_field].hist(bins=30, color='skyblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# Boxplot for the numeric field grouped by group_field (only if it's categorical with limited unique values)
if pd.api.types.is_object_dtype(df[group_field]) and df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 6))
    df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we explored the FAIR^2 dataset package:

- Loaded Croissant metadata and listed record sets and fields by their `@id`s.
- Extracted records into DataFrames for each record set.
- Demonstrated EDA by filtering, normalizing, and grouping data using key numeric and categorical fields.
- Visualized the distribution of a principal numeric field, helping characterize protein abundance features in the data.

These steps can form a basis for deeper biological analysis, machine learning workflows, or further data curation as needed.